In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import polars as pl
from collections import Counter
from tqdm import tqdm
import unicodedata
import re
from functools import partial
import random

In [3]:
df = pl.read_csv("urbandict-word-defs.csv", has_header=True, truncate_ragged_lines=True, infer_schema_length=0)

df_clean = df.filter(
    pl.col("definition").is_not_null() & 
    pl.col("word").is_not_null()
)

df_clean = df_clean.with_columns([
    pl.col("up_votes").cast(pl.Int64, strict=False),
    pl.col("down_votes").cast(pl.Int64, strict=False)
])

df_clean = df_clean.drop_nulls(subset=["up_votes", "down_votes"])


In [4]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normaliseString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s) # Add space before punctuation
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s) # Keep only letters and !?
    return s.strip()

In [5]:
class WordTokenizer:
    def __init__(self, texts=None):
        self.special_tokens = ["<pad>", "<unk>", "<sos>", "<eos>"]

        self.word2index = {tok: i for i, tok in enumerate(self.special_tokens)}
        self.index2word = {i: tok for i, tok in enumerate(self.special_tokens)}
        self.n_words = len(self.special_tokens)
        if texts is not None:
            self.build_vocab(texts)

    def build_vocab(self, texts):
        for text in texts:
            normalised = normaliseString(text)
            for word in normalised.split():
                self.add_word(word)
            

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.n_words += 1

    def encode(self, text):
        normalised = normaliseString(text)
        tokens = normalised.split()
        ids = [self.word2index.get(tok, self.word2index["<unk>"]) for tok in tokens]
        return [self.word2index["<sos>"]] + ids + [self.word2index["<eos>"]]
    
    def decode(self, ids):
        words = [self.index2word.get(id, "<unk>") for id in ids]
        return " ".join([w for w in words if w not in self.special_tokens])

    def __call__(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        return [self.encode(text) for text in texts]
    
    @property
    def pad_token_id(self):
        return self.word2index["<pad>"]
    
    @property
    def vocab_size(self):
        return self.n_words

In [6]:
class SimpleDataset(Dataset):
    def __init__(self, inputs, outputs, tokenizer, batch_size=1):

        self.input_ids = tokenizer(inputs)
        self.output_ids = tokenizer(outputs)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {"input_ids": torch.tensor(self.input_ids[idx],dtype=torch.long), "output_ids": torch.tensor(self.output_ids[idx],dtype=torch.long)}

In [7]:
def collate_fn(batch, pad_token_id):
    batch_input_ids = [item["input_ids"] for item in batch]
    batch_output_ids = [item["output_ids"] for item in batch]

    padded_inputs = nn.utils.rnn.pad_sequence(
        batch_input_ids, 
        padding_value=pad_token_id,
        batch_first=True
    )

    padded_outputs = nn.utils.rnn.pad_sequence(
        batch_output_ids, 
        padding_value=pad_token_id,
        batch_first=True
    )
    return {"input_ids": padded_inputs, "output_ids": padded_outputs}

In [8]:
class EncoderRNN(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.gru = nn.GRU(embedding_size, hidden_size, batch_first=True)

    def forward(self, input):
        embedding = self.embedding(input)
        outputs, hidden = self.gru(embedding)

        # outputs: [batch_size, seq_length, hidden_size]
        # hidden: [1, batch_size, hidden_size]
        return outputs, hidden

In [9]:
class Attention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, encoder_outputs, hidden):

        hidden = hidden.permute(1,0,2) # [batch_size, 1, hidden_size]
        attention_scores = torch.bmm(hidden, encoder_outputs.transpose(1,2)) # [batch_size, 1, seq_length]
        attention_weights = F.softmax(attention_scores,dim=2) # [batch_size, 1, seq_length]
        attention_vector = torch.bmm(attention_weights, encoder_outputs) # [batch_size, 1, hidden_size]

        return attention_vector, attention_weights

In [10]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.W_q = nn.Linear(hidden_size, hidden_size)
        self.W_k = nn.Linear(hidden_size, hidden_size)
        self.W_v = nn.Linear(hidden_size, hidden_size)

        self.scale = hidden_size ** 0.5

    def forward(self, encoder_outputs, hidden):

        hidden = hidden.permute(1,0,2) # [batch_size, 1, hidden_size]

        Q = self.W_q(hidden)
        K = self.W_k(encoder_outputs)
        V = self.W_v(encoder_outputs)

        attention_scores = torch.bmm(Q, K.transpose(1,2)) / self.scale # [batch_size, 1, seq_length]
        attention_weights = F.softmax(attention_scores,dim=2) # [batch_size, 1, seq_length]
        attention_vector = torch.bmm(attention_weights, V) # [batch_size, 1, hidden_size]

        return attention_vector, attention_weights

In [16]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size, attention):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.attention = attention
        self.gru = nn.GRU(embedding_size + hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, -1)

    def forward(self, encoder_outputs, hidden, input):
        embedding = self.embedding(input)
        attention_vector, _ = self.attention(encoder_outputs, hidden)
        rnn_input = torch.concat([embedding, attention_vector], dim=2)

        output, hidden = self.gru(rnn_input)
        output = self.out(output)
        return output, hidden

In [11]:
definitions  = df_clean["definition"].to_list()[:10000]
words  = df_clean["word"].to_list()[:10000]

tokenizer  = WordTokenizer(definitions  + words )
dataset  = SimpleDataset(definitions , words , tokenizer )

dataloader  = DataLoader(dataset , collate_fn=partial(collate_fn, pad_token_id=tokenizer .pad_token_id), batch_size=64, shuffle=True)
print("Vocab size: ", tokenizer.vocab_size)

Vocab size:  19251
